# 5. Business Insights

Translate the analysis outputs into concise findings and action-oriented recommendations.

## Environment Setup

Define project paths and display settings for the insights workflow.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets" / "screenshots"
DATA_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## Analysis Dataset Load

Import the prepared dataset used to generate the final business insights.

In [ ]:
eda_path = DATA_DIR / "cleaned_data_for_EDA.csv"
df = pd.read_csv(eda_path, parse_dates=["Order Date", "Ship Date"])
df.head()

## Executive KPI Table

Build a concise KPI table covering sales, profit, margin, and top performance areas.

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()

summary = pd.DataFrame({
    "KPI": [
        "Total Sales",
        "Total Profit",
        "Profit Margin",
        "Top Region by Sales",
        "Top Category by Profit",
        "Most Profitable Sub-Category",
        "Highest Loss Area",
    ],
    "Result": [
        f"${total_sales:,.2f}",
        f"${total_profit:,.2f}",
        f"{total_profit / total_sales:.2%}",
        df.groupby("Region")["Sales"].sum().idxmax(),
        df.groupby("Category")["Profit"].sum().idxmax(),
        df.groupby("Sub-Category")["Profit"].sum().idxmax(),
        df.groupby("Sub-Category")["Profit"].sum().idxmin(),
    ],
})
summary

## Reusable Performance Tables

Create grouped summaries for categories, sub-categories, regions, and customer segments.

In [ ]:
def group_summary(column):
    return (
        df.groupby(column, as_index=False)
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Average_Discount=("Discount", "mean"))
        .assign(
            Sales_Share=lambda x: x["Sales"] / total_sales,
            Profit_Share=lambda x: x["Profit"] / total_profit,
            Profit_Margin=lambda x: x["Profit"] / x["Sales"],
        )
        .sort_values("Profit", ascending=False)
    )

category = group_summary("Category")
subcategory = group_summary("Sub-Category")
region = group_summary("Region")
segment = group_summary("Segment")

## Business Findings

Convert the strongest patterns in the data into stakeholder-ready findings with supporting numbers.

In [ ]:
insights = pd.DataFrame([
    {
        "Insight": "Technology is the strongest category",
        "Evidence": f"Technology generated ${category.loc[category['Category'] == 'Technology', 'Profit'].iloc[0]:,.2f} profit, equal to {category.loc[category['Category'] == 'Technology', 'Profit_Share'].iloc[0]:.1%} of total profit.",
    },
    {
        "Insight": "Copiers are the top profit driver",
        "Evidence": f"Copiers produced ${subcategory.loc[subcategory['Sub-Category'] == 'Copiers', 'Profit'].iloc[0]:,.2f} profit with a {subcategory.loc[subcategory['Sub-Category'] == 'Copiers', 'Profit_Margin'].iloc[0]:.1%} margin.",
    },
    {
        "Insight": "Tables are the largest loss-making area",
        "Evidence": f"Tables generated ${subcategory.loc[subcategory['Sub-Category'] == 'Tables', 'Profit'].iloc[0]:,.2f} profit on ${subcategory.loc[subcategory['Sub-Category'] == 'Tables', 'Sales'].iloc[0]:,.2f} sales.",
    },
    {
        "Insight": "West is the best-performing region",
        "Evidence": f"West contributed ${region.loc[region['Region'] == 'West', 'Sales'].iloc[0]:,.2f} sales and ${region.loc[region['Region'] == 'West', 'Profit'].iloc[0]:,.2f} profit.",
    },
    {
        "Insight": "Consumer is the largest customer segment",
        "Evidence": f"Consumer customers generated ${segment.loc[segment['Segment'] == 'Consumer', 'Sales'].iloc[0]:,.2f} sales, equal to {segment.loc[segment['Segment'] == 'Consumer', 'Sales_Share'].iloc[0]:.1%} of total sales.",
    },
])
insights

## Recommended Actions

Summarize practical actions focused on margin improvement and profitable growth.

In [ ]:
recommendations = pd.DataFrame({
    "Recommendation": [
        "Reduce discounting on Tables and other weak Furniture products.",
        "Increase promotional focus on Technology, especially Copiers.",
        "Review Central region discount policy to improve margin quality.",
        "Use Consumer segment revenue strength for targeted retention and upsell campaigns.",
        "Protect West region performance by maintaining inventory and service levels for high-margin products.",
    ]
})
recommendations